# Real-Study Compatibility Demo · DeepSeek + P-VCNA-003 Library

**Midterm experiment (not a clinical validation):** read a target-study PDF from Google Drive, choose 1–2 Tables, use **DeepSeek** for evidence-backed specification extraction, retrieve a source template from the **P-VCNA-003** library, inspect the match and the proposed adaptation, and export audit files.

- This Notebook uses real API calls in Step 2 (**no fixture fallback**).
- Source Library files and teammate metadata are read-only.
- Stage B.1: source-verified Bookmark DOCX adaptation is available for simple text edits only. Unmapped headers, statistical rows, footnotes, and unsupported structure remain blocked. Successful outputs are **draft_needs_review**, never clinical approval.
- **A target Study different from P-VCNA-003 is preferred** for evaluating reuse. P-VCNA-003's own SAP is incomplete and must not be treated as an independent end-to-end evaluation.
- Avoid uploading private or regulated data unless your team has permission to send the selected text excerpts to DeepSeek.

**Step 2/3 update:** verified enumeration quotes may support multiple explicitly mentioned values; retrieval ranks Table family/title/statistics/methods/population. Scores are heuristic; known endpoint changes require review.

**Step 2 targeted completion:** if verified required fields are missing, the agent makes at most one additional DeepSeek extraction call, restricted to target-SAP evidence. Only values with verified quotations are retained. **Step 3 review gate:** best-ranked template is not automatically usable when source and target populations or endpoints differ.


## 0 · Install the package from GitHub (no ZIP upload)

In [ ]:
from pathlib import Path
import os, sys, subprocess
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
PROJECT_ROOT = Path('/content/tfl-generation-agent')
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError('Project path already exists but is not a Git checkout')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_ROOT}[notebook]'])
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.tools.llm_client import DeepSeekJSONClient
import pandas as pd
from IPython.display import display, FileLink
print('Agent package loaded:', PROJECT_ROOT)


## 1 · Audit the real source Library (no API call)

This confirms that GitHub contains the study-scoped v2.4 Metadata and the real DOCX files. The optional `Keyword_Index.json` is **not required**: the new loader uses keywords embedded in each template record.

In [ ]:
LIBRARY_PATH = PROJECT_ROOT / 'library' / 'studies' / 'P-VCNA-003'
library = load_library(LIBRARY_PATH)
counts = pd.Series([t['identity']['output_type'] for t in library['templates']]).value_counts()
print('Library:', library['library_id'])
print('Format:', library['format'], '| Index mode:', library['index_mode'])
print('Total templates:', len(library['templates']))
display(counts.rename_axis('Type').reset_index(name='Count'))
assert len(library['templates']) == 64, 'Check that the entire P-VCNA-003 library was cloned.'
print('All listed DOCX paths resolved successfully.')


## 2 · Load the target SAP from Google Drive (persistent input)

1. In Google Drive, create `My Drive/Colab Notebooks/agent/midterm demo/`.
2. Upload your target SAP `SAP_001_00.pdf` to that folder **once**. The file is not added to GitHub.
3. Run the next cell, authorize Google Drive, and check that `File exists: True` appears.

You may use a different folder or PDF name: update `DRIVE_SAP_RELATIVE_PATH` below to the **relative path under My Drive**. For this PDF, the statistical SAP body is on physical pages 4–21; Content List and shell pages should not be used as statistical evidence.

**Privacy:** The provided SAP is marked confidential. Only place it on Drive and submit its excerpts to DeepSeek if you have the appropriate authorization.


In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Drive once per fresh Colab runtime (follow the Google authorization prompt).
drive.mount("/content/drive")

# Relative to "My Drive" in Google Drive; change only this line if needed.
DRIVE_SAP_RELATIVE_PATH = "Colab Notebooks/agent/midterm demo/SAP_001_00.pdf"
SAP_PATH = Path("/content/drive/MyDrive") / DRIVE_SAP_RELATIVE_PATH

if not SAP_PATH.is_file():
    folder = SAP_PATH.parent
    available = sorted(p.name for p in folder.glob("*.pdf")) if folder.is_dir() else []
    raise FileNotFoundError(
        f"Cannot find target SAP: {SAP_PATH}\n"
        f"Google Drive folder exists: {folder.is_dir()}\n"
        f"PDF files found in folder: {available}\n"
        "Upload the PDF to your configured Drive folder, or update "
        "DRIVE_SAP_RELATIVE_PATH to its actual location."
    )

print("SAP loaded from Google Drive:", SAP_PATH)
print("File exists:", SAP_PATH.is_file())
print("File size (MB):", round(SAP_PATH.stat().st_size / (1024**2), 2))


## 3 · Configure 1–2 target Tables and optional PDF page range

Replace the example TFL IDs and titles with the **exact TFL names in your SAP PDF**, not with the P-VCNA-003 source template ID. These targets are deliberately selected manually; this MVP does **not** assert automatic extraction of the full Content List.

`SAP_BODY_PAGE_RANGE=(first,last)` uses **physical PDF page numbers (1-indexed)**, not printed page labels. This is strongly recommended when the same PDF contains existing shells. Set `None` only when you cannot identify the SAP body range; otherwise template text could be mistaken for analysis evidence.

In [ ]:
TARGET_TABLES = [
    {"tfl_id": "14.1.3.1", "title": "Demographic and Baseline Characteristics – Safety Analysis Set"},
    # {"tfl_id": "14.2.3.1", "title": "Summary of Progression Free Survival – ITT Analysis Set"},
]
# These are physical (1-based) pages of SAP_001_00.pdf, excluding the
# embedded Content List (pages 23–26) and source TFL shells (pages 27+).
SAP_BODY_PAGE_RANGE = (4, 21)

SCHEMA_PATH = PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
OUTPUT_DIR = PROJECT_ROOT / "output" / "real_study_deepseek"
assert SAP_PATH.is_file(), f"SAP not found: {SAP_PATH}"
assert TARGET_TABLES and len(TARGET_TABLES) <= 2
for target in TARGET_TABLES:
    assert target["tfl_id"] and target["title"]
print("Targets:", TARGET_TABLES)
print("SAP:", SAP_PATH)
print("SAP body page restriction:", SAP_BODY_PAGE_RANGE)

# Optional, separately human-reviewed source-template mappings for Step 4.
# Leave EMPTY to preserve Step 3 compatibility gates. See STEP4_BOOKMARK_ADAPTATION.md.
REVIEWED_ADAPTATIONS = {}


## 4 · Connect DeepSeek using **Colab Secrets**

In the Colab left sidebar, open **Secrets** (key icon), create `DEEPSEEK_API_KEY`, enter a **newly rotated** DeepSeek key, and enable notebook access. Do **not** paste the key into a cell, README, or GitHub file.

Every target Table uses **two real DeepSeek calls** (query planning + initial evidence extraction), plus **at most one additional completion call** if required evidence fields are missing; requests may incur charges. A request error is reported, never replaced with synthetic data.

In [ ]:
from google.colab import userdata
MODEL = 'deepseek-flash'
key = userdata.get('DEEPSEEK_API_KEY')
if not key:
    raise RuntimeError('Missing DEEPSEEK_API_KEY in Colab Secrets')
llm = DeepSeekJSONClient(model=MODEL, api_key=key)
del key
print('DeepSeek client initialized; key not printed.')


## 5 · Run Step 1–5 for each selected Table

Watch actual extracted SAP evidence, missing-field completion audit, Library match scores (heuristics, **not** model probabilities), the best-ranked candidate versus the safely selected template, and the modification plan. `needs_review` is an informative **compatibility limitation**, not a generated shell.

If the source SAP lacks statistical definitions (or source pages are missing), Step 2 may be `blocked`. This is a legitimate failure case.
**Step 4:** An optional reviewed mapping (configured above) can permit a DRAFT from a candidate that Step 3 flagged for source/target population differences. Without such a mapping, selection remains blocked. A high retrieval score alone never grants adaptation permission.


In [ ]:
import json
from docx import Document
summaries = []
sessions = {}
for target in TARGET_TABLES:
    tid, title = target['tfl_id'], target['title']
    print('\n' + '='*65)
    print('TARGET:', tid, title)
    safe_id = ''.join(c if c.isalnum() or c in '._-' else '_' for c in tid)
    session = TFLSession(
        sap_path=SAP_PATH, library_path=LIBRARY_PATH,
        schema_path=SCHEMA_PATH, prompts_dir=PROMPTS_DIR,
        output_dir=OUTPUT_DIR / safe_id, llm=llm,
        tfl_id=tid, title=title,
        sap_body_page_range=SAP_BODY_PAGE_RANGE,
        reviewed_adaptation=REVIEWED_ADAPTATIONS.get(tid),
        allow_review=tid in REVIEWED_ADAPTATIONS,
    )
    sessions[tid] = session
    try:
        parsed = session.step1_parse()
        print('STEP 1 / Parsed:', json.dumps(parsed, indent=2, ensure_ascii=False))
        spec = session.step2_build_specification()
        print('STEP 2 / Readiness:', spec['readiness'])
        completion = (session.spec_debug or {}).get('completion', {})
        print('STEP 2 / Targeted completion attempted:', completion.get('attempted', False))
        print('STEP 2 / Missing before:', completion.get('missing_before', []))
        print('STEP 2 / Recovered with verified SAP citations:', completion.get('verified_filled_fields', []))
        print('STEP 2 / Still missing:', completion.get('missing_after', []))
        display(pd.DataFrame([spec['statistical_requirements']]).T.rename(columns={0:'Extracted requirement'}))
        ev = spec.get('provenance', {}).get('evidence', [])
        print('SAP evidence blocks:', len(ev))
        if ev:
            display(pd.DataFrame([{'Page': e.get('page_numbers'), 'Section': e.get('section'),
                                   'Exact SAP quotation': e.get('exact_text', '')[:300]}
                                  for e in ev]).head(10))
        if spec.get('issues'):
            display(pd.DataFrame(spec['issues']).head(15))
        retrieved = session.step3_retrieve_templates(top_k=5)
        print('STEP 3 / Best-ranked source candidate:', retrieved.get('best_candidate_template_id'))
        print('STEP 3 / Retrieval status:', retrieved.get('status'))
        print('STEP 3 / Automatically usable template:', retrieved['selected_template_id'])
        if retrieved.get('reasons'):
            print('STEP 3 / Review or selection reasons:', '; '.join(retrieved['reasons']))
        display(pd.DataFrame(retrieved['candidates']))
        if retrieved['candidates']:
            top = retrieved['candidates'][0]
            print('TOP RANKED CANDIDATE (not necessarily auto-selected):', top['template_id'])
            print('Heuristic score:', top['score'])
            print('Ranking rationale:', top.get('explanation', ''))
            if top.get('requires_endpoint_review'):
                print('MANUAL REVIEW REQUIRED: source and target endpoints differ.')
            if top.get('population_change_required'):
                print('Source analysis population differs: adaptation required.')
        if not session.selection and session.matches:
            print('Best-ranked template is RETRIEVED but NOT approved for modification.')
            print('Source template for inspection only (NOT generated output):')
            display(FileLink(session.matches[0]['source_path']))
        if session.selection:
            selected = session.selection
            print('Why this source template:', selected['explanation'])
            print('Source study:', selected.get('source_study_id'))
            print('Source DOCX (NOT generated output):')
            display(FileLink(selected['source_path']))
            notes = selected['template'].get('footnotes', [])
            print('Source template footnotes:', len(notes))
            if notes:
                display(pd.DataFrame([{'Footnote': x.get('exact_text','')[:250],
                                       'Semantic review': x.get('semantic_extraction_status')}
                                      for x in notes]).head(5))
        plan = session.preview_modification_plan()
        print('STEP 4 / Modification feasible:', plan['safe_to_generate'])
        print('Editing mode:', plan.get('editing_mode', 'mvp_placeholders'))
        print('Reason:', plan.get('reason', 'See replacement plan below.'))
        if plan.get('issues'):
            print('BOOKMARK REVIEW GATES:', plan['issues'][:10])
        if plan.get('edits'):
            display(pd.DataFrame(plan['edits'])[['bookmark', 'source_role', 'expected_source_text', 'new_text', 'reason']].head(15))
        elif plan.get('safe_to_generate'):
            display(pd.DataFrame(list(plan.get('replacements', {}).items()), columns=['Placeholder','Replacement']))
        generation = session.step4_generate_shell()
        print('STEP 4 / Generation status:', generation['status'])
        if generation['issues']:
            print('Generation issues:', generation['issues'][:5])
        report = session.step5_validate_and_export()
        print('STEP 5 / Validation status:', report['status'])
        print('Spec evidence/schema valid:', report['schema_and_evidence']['schema_valid'])
        if session.shell_path:
            display(FileLink(session.shell_path, result_html_prefix='Download generated DOCX: '))
        print('Saved audit output:', session.output_dir)
        summaries.append({'TFL': tid, 'Status': report['status'],
                          'Template': session.selection['template_id'] if session.selection else None,
                          'Spec readiness': spec['readiness'],
                          'Generated DOCX': bool(session.shell_path)})
    except Exception as exc:
        # Do not claim success or silently fall back to a fixture.
        print('FAILED:', type(exc).__name__, str(exc))
        summaries.append({'TFL': tid, 'Status': 'failed', 'Template': None,
                          'Spec readiness': None, 'Generated DOCX': False})
print('\nCOMPATIBILITY TEST SUMMARY')
display(pd.DataFrame(summaries))


## 6 · Export audit artifacts and preserve demo screenshots

All outputs are under `/content/tfl-generation-agent/output/real_study_deepseek/`. This output directory is **temporary** in Colab. The **input SAP on Google Drive persists** after the runtime ends. Download the ZIP or copy it to a separately authorized private Drive location. Do not commit real SAP text/evidence or private API data back to the public GitHub repository.

In [ ]:
import shutil
from google.colab import files
archive_base = Path('/content/tfl_real_study_audit')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR))
print('Audit archive:', archive_path, 'size:', archive_path.stat().st_size)
files.download(str(archive_path))
